In [28]:
import numpy as np
import pandas as pd
import talib
import yfinance as yf
from backtesting import Backtest, Strategy


class ORBGoldStrategy(Strategy):

  def init(self):
    # Cast Volume to float for TA-Lib
    volume_float = self.data.Volume.astype(float)
    self.vol_ema = self.I(talib.EMA, volume_float, timeperiod=20)

    # State variables for daily tracking
    self.current_day = None
    self.orb_high = 0
    self.orb_low = 0
    self.bar_count_today = 0
    self.trade_taken_today = False
    self.vol_condition_met = False

  def next(self):
    current_time = self.data.index[-1]
    current_date = current_time.date()

    # --- 1. Daily Reset ---
    if self.current_day != current_date:
      if self.position:
        self.position.close()

      self.current_day = current_date
      self.bar_count_today = 0
      self.trade_taken_today = False
      self.vol_condition_met = False
      self.orb_high = 0
      self.orb_low = 0

    self.bar_count_today += 1

    # End of day square off check
    if self.bar_count_today > 26:
      if self.position:
        self.position.close()
      return

    # --- 2. Mark High and Low of 1st 15-min Candle ---
    if self.bar_count_today == 1:
      self.orb_high = self.data.High[-1]
      self.orb_low = self.data.Low[-1]
      return

    if self.trade_taken_today:
      return

    # --- 3. Second Candle Logic ---
    if self.bar_count_today == 2:
      op = self.data.Open[-1]
      if op > self.orb_high:
        self.buy()
        self.trade_taken_today = True
        return
      elif op < self.orb_low:
        self.sell()
        self.trade_taken_today = True
        return

    # --- 4. Within Range Logic (Candles 2 to 5) ---
    if 2 <= self.bar_count_today <= 5:
      if self.data.Volume[-1] > self.vol_ema[-1]:
        self.vol_condition_met = True

      if self.data.Close[-1] > self.orb_high and self.vol_condition_met:
        self.buy()
        self.trade_taken_today = True
        return
      elif self.data.Close[-1] < self.orb_low and self.vol_condition_met:
        self.sell()
        self.trade_taken_today = True
        return

    # If 4 candles pass with no valid move, lock trading for the day
    if self.bar_count_today == 6 and not self.trade_taken_today:
      self.trade_taken_today = True

    # --- 5. Programmatic Stop Loss (Opposite end of 15-min range) ---
    if self.position:
      if self.position.is_long:
        if self.data.Low[-1] <= self.orb_low:
          self.position.close()
      elif self.position.is_short:
        if self.data.High[-1] >= self.orb_high:
          self.position.close()


# --- Data Download & Backtest Execution ---
data = yf.download("GC=F", interval="15m", period="60d")

if isinstance(data.columns, pd.MultiIndex):
  data.columns = data.columns.droplevel(1)

data = data[["Open", "High", "Low", "Close", "Volume"]].dropna()

# Fixed flat commission of $60 per trade
bt = Backtest(
    data,
    ORBGoldStrategy,
    cash=100000,
    commission=lambda size, price: 60.0 / (abs(size) * price),
    exclusive_orders=True,
)
output = bt.run()
print(output)
bt.plot()

C:\Users\DELL\AppData\Local\Temp\ipykernel_15660\1384328181.py:97: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("GC=F", interval="15m", period="60d")
[*********************100%***********************]  1 of 1 completed


Backtest.run:   0%|          | 0/4469 [00:00<?, ?bar/s]

Start                     2026-08-09 22:00...
End                       2026-10-16 20:45...
Duration                     67 days 22:45:00
Exposure Time [%]                    23.52417
Equity Final [$]                  97833.53127
Equity Peak [$]                    100800.796
Commissions [$]                       0.05418
Return [%]                           -2.16647
Buy & Hold Return [%]                -4.54109
Return (Ann.) [%]                    -9.07761
Volatility (Ann.) [%]                 7.81295
CAGR [%]                             -7.80197
Sharpe Ratio                         -1.16187
Sortino Ratio                        -1.41744
Calmar Ratio                         -1.72442
Alpha [%]                            -2.06216
Beta                                  0.02297
Max. Drawdown [%]                    -5.26416
Avg. Drawdown [%]                    -1.01411
Max. Drawdown Duration       66 days 18:45:00
Avg. Drawdown Duration       11 days 06:35:00
# Trades                          

GridPlot(id='p2201', ...)

In [29]:
output['_trades']

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,"Entry_EMA(V,20)","Exit_EMA(V,20)"
0,22,22,47,4387.700195,4403.200195,None,None,340.998759,0.001241,0.003533,2026-08-10 03:30:00+00:00,2026-08-10 09:45:00+00:00,0 days 06:15:00,None,749.236108,1082.572374
1,22,102,127,4473.100098,4422.899902,None,None,-1104.405523,0.001226,-0.011223,2026-08-11 00:30:00+00:00,2026-08-11 06:45:00+00:00,0 days 06:15:00,None,980.697703,1581.131366
2,22,197,219,4450.200195,4454.399902,None,None,92.392330,0.001225,0.000944,2026-08-12 01:15:00+00:00,2026-08-12 06:45:00+00:00,0 days 05:30:00,None,914.162463,1041.504778
3,22,286,311,4487.600098,4431.299805,None,None,-1238.607669,0.001223,-0.012546,2026-08-13 00:30:00+00:00,2026-08-13 06:45:00+00:00,0 days 06:15:00,None,1152.007811,1310.735877
4,-22,378,403,4395.500000,4389.000000,None,None,142.998758,0.001242,0.001479,2026-08-14 00:30:00+00:00,2026-08-14 06:45:00+00:00,0 days 06:15:00,None,1064.821386,913.449153
5,22,471,496,4445.799805,4452.500000,None,None,147.403071,0.001226,0.001507,2026-08-17 00:30:00+00:00,2026-08-17 06:45:00+00:00,0 days 06:15:00,None,844.560610,839.543903
6,21,563,588,4487.799805,4452.600098,None,None,-739.195126,0.001278,-0.007843,2026-08-18 00:30:00+00:00,2026-08-18 06:45:00+00:00,0 days 06:15:00,None,913.125560,871.498738
7,-21,750,772,4551.299805,4550.399902,None,None,18.896694,0.001256,0.000198,2026-08-20 01:15:00+00:00,2026-08-20 06:45:00+00:00,0 days 05:30:00,None,1638.493706,1173.980063
8,-20,931,956,4661.600098,4699.500000,None,None,-757.999329,0.001282,-0.008130,2026-08-24 00:30:00+00:00,2026-08-24 06:45:00+00:00,0 days 06:15:00,None,1553.646886,1477.349365
9,20,1024,1048,4751.500000,4704.200195,None,None,-945.997363,0.001269,-0.009955,2026-08-25 00:45:00+00:00,2026-08-25 06:45:00+00:00,0 days 06:00:00,None,1527.218246,1461.757423


In [31]:
min_loss =output['_trades']['PnL'][output['_trades']['PnL']<0].max() 
mean_loss=output['_trades']['PnL'][output['_trades']['PnL']<0].mean()
max_loss = output['_trades']['PnL'][output['_trades']['PnL']<0].min()
print('Max loss:',max_loss)
print('Avg loss:',mean_loss)
print('Min loss:',min_loss)

Max loss: -1238.6076685043447
Avg loss: -520.1376655606961
Min loss: -22.003240388603846
